In [ ]:
import os
import json
from PIL import Image
import pandas as pd
import re

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
file_path = '/content/gdrive/MyDrive/ECE1786_project/product_data_full.csv'
df = pd.read_csv(file_path)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   img_dir  200 non-null    int64  
 1   price    200 non-null    float64
 2   label    200 non-null    object 
dtypes: float64(1), int64(1), object(1)
memory usage: 4.8+ KB


In [ ]:
base_dir = "/content/gdrive/MyDrive/ECE1786_project/data"

In [ ]:
def parse_to_json_with_re(input_str):
    # Use regex to extract values
    color_match = re.search(r"Color:\s*([^/]+)", input_str)  # Match until the `/`
    if not color_match:
        raise ValueError(f"Unable to extract 'color' from input: {input_str}")
    color = color_match.group(1).strip()

    length_match = re.search(r"L:\s*([\d.]+)", input_str)
    width_match = re.search(r"W:\s*([\d.]+)", input_str)
    height_match = re.search(r"H:\s*([\d.]+)", input_str)
    unit_match = re.search(r",\s*(\w+)\s*/", input_str)
    price_match = re.search(r"Price:\s*([\d.]+)", input_str)

    if not all([length_match, width_match, height_match, unit_match, price_match]):
        raise ValueError(f"Unable to extract dimensions or price from input: {input_str}")

    length = float(length_match.group(1))
    width = float(width_match.group(1))
    height = float(height_match.group(1))
    unit = unit_match.group(1)
    price = float(price_match.group(1))

    # Create a dictionary for the JSON object
    data = {
        "color": color,
        "length": length,
        "width": width,
        "height": height,
        "unit": unit,
        "price": price
    }

    # Convert to single-line JSON string
    json_data = json.dumps(data)
    return json_data

In [ ]:
# Function to generate JSONL data
def create_combined_jsonl(df, base_dir, output_file):
    combined_data = []

    for i, row in df.iterrows():
        dir_path = os.path.join(base_dir, str(row['img_dir']))
        text_file = [f for f in os.listdir(dir_path) if f.endswith('.txt')]
        image_files = [f for f in os.listdir(dir_path) if f.endswith('.jpg')]
        label = parse_to_json_with_re(row['label'])

        # print(text_file)
        # Read the product title from the text file
        if text_file:
            with open(os.path.join(dir_path, text_file[0]), 'r', encoding='utf-8', errors='ignore') as f:
                product_title = f.read().strip()
        else:
            product_title = "Unknown Title"

        # Get image paths and dimensions
        image_paths = []
        widths, heights = [], []
        for image_file in image_files:
            image_path = os.path.join(str(row['img_dir']), image_file)
            image_paths.append(image_path)

            # Retrieve image dimensions
            image_full_path = os.path.join(base_dir, image_path)
            with Image.open(image_full_path) as img:
                widths.append(img.width)
                heights.append(img.height)

        # Create a combined JSONL entry
        if len(image_paths) == 1:
            prompt_sg = (
            f'Product Image: <image>\n'
            f'Product Title: {product_title}\n'
            f'Product Price: {row["price"]}\n'
            f'Task: Analyze the provided product image and title to accurately extract key features such as color, dimensions (length, width, height), unit of measurement, and price. Ensure the output is formatted as a single-line JSON string.\n'
            f'Output format: {{"color": "[COLOR]", "length": [LENGTH], "width": [WIDTH], "height": [HEIGHT], "unit": "[UNIT]", "price": [PRICE]}}.\n'
            f'Features: '
            )
            combined_data.append({
                "id": row['img_dir'],
                "image": image_paths[0],
                "width": widths[0],
                "height": heights[0],
                "conversations": [
                    {"from": "human", "value": prompt_sg},
                    {"from": "gpt", "value": label}
                ]
            })
        elif len(image_paths) > 1:
            prompt_mt = (
            f'Product Image 1: <image>\n'
            f'Product Image 2: <image>\n'
            f'Product Title: {product_title}\n'
            f'Product Price: {row["price"]}\n'
            f'Task: Analyze the provided product images and title to accurately extract key features such as color, dimensions (length, width, height), unit of measurement, and price. Use the information from both images to ensure accuracy. Ensure the output is formatted as a single-line JSON string.\n'
            f'Output format: {{"color": "[COLOR]", "length": [LENGTH], "width": [WIDTH], "height": [HEIGHT], "unit": "[UNIT]", "price": [PRICE]}}.\n'
            f'Features: '
            )
            combined_data.append({
                "id": row['img_dir'],
                "image": image_paths,
                "width_list": widths,
                "height_list": heights,
                "conversations": [
                    {"from": "human", "value": prompt_mt},
                    {"from": "gpt", "value": label}
                ]
            })
        if i % 10 == 0:
            print(f"Processed {i} rows")

    # Write to a single JSONL file
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in combined_data:
            json.dump(entry, f)
            f.write('\n')

In [ ]:
output_jsonl_file = "/content/gdrive/MyDrive/ECE1786_project/product_data_full.jsonl"

In [ ]:
create_combined_jsonl(df, base_dir, output_jsonl_file)

Processed 0 rows
Processed 10 rows
Processed 20 rows
Processed 30 rows
Processed 40 rows
Processed 50 rows
Processed 60 rows
Processed 70 rows
Processed 80 rows
Processed 90 rows
Processed 100 rows
Processed 110 rows
Processed 120 rows
Processed 130 rows
Processed 140 rows
Processed 150 rows
Processed 160 rows
Processed 170 rows
Processed 180 rows
Processed 190 rows


In [ ]:
file_path = '/content/gdrive/MyDrive/ECE1786_project/user_query.xlsx'
df_user_query = pd.read_excel(file_path)
# df_user_query.drop(index=0, inplace=True)
# df_user_query.reset_index(drop=True, inplace=True)

In [ ]:
df_user_query.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   query   131 non-null    object
 1   label   131 non-null    object
dtypes: object(2)
memory usage: 2.2+ KB


In [ ]:
df_user_query.head(1)

,query,label
0,"I want a light-colored coffee table, roughly 3...","{""color"": ""Light"", ""length"": ""3 ft"", ""price"": 75}"


In [ ]:
user_data = []
for i, row in df_user_query.iterrows():
  prompt_user = (
    f'Task: Extract key product features from the user\'s query. The features may include any of the following keys: \'color\', \'length\', \'width\', \'height\', \'price\', depending on what is mentioned in the query.\n'
    f'Query: {row["query"]}\n'
    f'Expected Output: Provide a JSON object containing only the relevant keys and their corresponding values based on the query.\n'
    f'Example Query: Can you find me a black study desk, about 48 inches tall? My budget is $200.\n'
    f'Example Output: {{"color": "Black", "height": "48 inch", "price": 200}}\n'
    f'Features: '
  )
  label = row['label']
  user_data.append({
      "id": i,
      "conversations": [
          {"from": "human", "value": prompt_user},
          {"from": "gpt", "value": labe2l}
      ]
  })

In [ ]:
# Write to a single JSONL file
output_file = "/content/gdrive/MyDrive/ECE1786_project/user_data.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in user_data:
        json.dump(entry, f)
        f.write('\n')

In [ ]:
# Define the meta information for the dataset
meta_data = {
    "dataset-1": {
        "root": "/content/gdrive/MyDrive/ECE1786_project/data",  # Base path to your dataset (update accordingly)
        "annotation": "/content/gdrive/MyDrive/ECE1786_project/product_data_full.jsonl",  # Path to the JSONL annotation file
        "data_augment": False,  # Set to True if data augmentation is needed
        "repeat_time": 1,  # Number of times the dataset is repeated
        "length": 200  # Replace with the actual number of samples in the dataset
    },
    "dataset-2": {
        "root": "/content/gdrive/MyDrive/ECE1786_project/data",  # Base path to your dataset (update accordingly)
        "annotation": "/content/gdrive/MyDrive/ECE1786_project/user_data.jsonl",  # Path to the JSONL annotation file
        "data_augment": False,  # Set to True if data augmentation is needed
        "repeat_time": 1,  # Number of times the dataset is repeated
        "length": 131  # Replace with the actual number of samples in the dataset
    }
}

# Save the meta information as a JSON file
meta_file_path = "/content/gdrive/MyDrive/ECE1786_project/meta_file_full.json"  # Update the path where you want to save the meta file
with open(meta_file_path, 'w', encoding='utf-8') as meta_file:
    json.dump(meta_data, meta_file, indent=4)

print(f"Meta file has been saved to {meta_file_path}")

Meta file has been saved to /content/gdrive/MyDrive/ECE1786_project/meta_file_full.json


In [ ]:
import torch
print("Torch:", torch.__version__)

Torch: 2.5.1+cu121
